# Lecture 7

## Data Exploration <a class="anchor" id="TOC"></a>

 - [Descriptive statistics with pandas:](#pandas)             
   - use of groupby for descriptives        
   - custom function                 
 - [Graphs with plotnine:](#graphs)                         
   - histogram                       
   - customize a plot              
   - kernel density                  
   - multiple geometries             
 - [Hypothesis testing with t.test](#ttest)  
 - [Association](#association)                      
   - scatter plot                    
   - bin-scatter:                    
       - equal distance              
       - equal number of obs         
   - correlation and covariance      
   - factors with ggplot             
                                     
Case-study:                           
- Billion Price Project: Online and Offline prices           
                                     
Dataset:                              
- billion-prices 

___

Import packages

In [ ]:
# Load tools for summaries and graphics; warning suppression hides messages, not
# problems.
import warnings

import numpy as np
import pandas as pd
from plotnine import *
from skimpy import skim

warnings.filterwarnings("ignore")

Import data

In [ ]:
# Read the local price data using the encoding needed to decode its text correctly.
from pathlib import Path


def find_data_dir():
    # Search here and then in parent folders for the project data.
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "data" / "raw"
        if candidate.exists():
            # Stop at the first matching data folder.
            return candidate
    return Path("data") / "raw"


DATA_DIR = find_data_dir()

bpp_original = pd.read_csv(DATA_DIR / "billion_prices.csv", encoding="latin-1")


Check variables

In [ ]:
# Inspect the first rows before calculating new variables.
bpp_original.head()


Create our key variable: price differences

In [ ]:
# Positive differences mean the online price exceeds the offline price for that row.
bpp_original["p_diff"] = bpp_original["price_online"] - bpp_original["price"]


## Descriptive statistics <a class="anchor" id="pandas"></a>

[back](#TOC)

Check all the variables in DataFrame by a quick built-in summary statistics

In [ ]:
# Get a first overview of numeric columns, including counts and distribution summaries.
bpp_original.describe()


Use skim, from the [`skimpy package`](https://aeturrell.github.io/skimpy/) for a nicer looking descriptive tabler

In [ ]:
# Inspect a richer overview of types, missing values, and distributions.
skim(bpp_original)

Compare key variables

In [ ]:
# Restrict the summary to the two prices and their difference.
bpp_original.filter(["price", "price_online", "p_diff"]).describe()


Put the descriptives into columns and variables into rows

In [ ]:
# Transpose the summary so variables appear as rows rather than columns.
bpp_original.filter(["price", "price_online", "p_diff"]).describe().transpose()


In [ ]:
# Apply the detailed summary to just the selected variables.
skim(bpp_original.filter(["price", "price_online", "p_diff"]))

Next let us check the price differences for each countries.

For this, you need to group the data and apply the required statistics to the appropriate columns

In [ ]:
# Calculate separate means and medians of the price difference for each country.
bpp_original.groupby("COUNTRY").agg(
    mean_price_diff=("p_diff", "mean"), median_price_diff=("p_diff", "median")
)


Lets say we are interested in the prices as well for each countries.

In [ ]:
# Reshape the three measures into one value column, then summarise each measure within
# country.
(
    bpp_original.melt(
        id_vars=["COUNTRY"], value_vars=["price", "price_online", "p_diff"]
    )
    .groupby(["COUNTRY", "variable"])
    .agg(Mean=("value", "mean"), Median=("value", "median"))
)


#### Task
1) filter the data to 2016 and check price difference the mean and median for each country

In [ ]:
# Filter to 2016 before computing country-level summaries.
(
    bpp_original.loc[bpp_original["year"] == 2016]
    .groupby("COUNTRY")
    .agg(Mean=("p_diff", "mean"), Median=("p_diff", "median"))
)


Add self created descriptive function to groupby().agg()

Lets check the 'range' as an external function to the descriptive

In [ ]:
# Define the observed range as maximum minus minimum and use it as a grouped summary.
def range_function(x):
    return x.max() - x.min()


bpp_original.groupby("COUNTRY").agg(
    mean_price_diff=("p_diff", "mean"),
    median_price_diff=("p_diff", "median"),
    range_price_diff=("p_diff", range_function),
)


Later we will discuss functions more in details!

### Graphs With plotnine <a class="anchor" id="graphs"></a>

[back](#TOC)

We use the plotnine package for visualization in Python. This package relates to the ggplot package in R. 

plotnine always has a `ggplot()` function and a `geom_*type*()` function added.

___

Check the empirical distribution:  histogram.

Before building a nice looking graph using plotnine, its worth to look at the histogram using thew built in pandas `hist()` function

In [ ]:
# Inspect the unfiltered price distribution before making sample restrictions.
bpp_original.price.hist()


It is clear: need to filter out some data!

In [ ]:
# Keep regular-price observations with both prices present and sale_online missing.
bpp = (
    bpp_original.loc[bpp_original["sale_online"].isnull()]
    .loc[bpp_original["price"].notnull()]
    .loc[bpp_original["price_online"].notnull()]
    .loc[bpp_original["PRICETYPE"] == "Regular Price"]
)


Check our newly created data:

In [ ]:
# Check the price summaries after applying the sample restrictions.
bpp.filter(["price", "price_online", "p_diff"]).describe()


Drop obvious errors: price is larger than $1000

In [ ]:
# Remove observations whose offline price is at least 1000; this changes the analysis
# sample.
bpp = bpp.loc[bpp["price"] < 1000]


In [ ]:
# Recheck the summaries to see the effect of the price restriction.
bpp.filter(["price", "price_online", "p_diff"]).describe()


In [ ]:
# Compare the filtered histogram with the earlier unfiltered distribution.
bpp.price.hist()


Make a nice histogram using plotnine

In [ ]:
# Divide the observed price range into 50 bins and count observations in each bin.
(
    ggplot(data=bpp)
    + geom_histogram(aes(x="price"), bins=50, fill="blue")
    + labs(x="Price", y="Count")
)


#### Role of number of bins (or binwidth)

Play with the number of Bins:

In [ ]:
# Use 50 bins as a reference when comparing how bin choices affect the picture.
# 1) approx ok
(
    ggplot(data=bpp)
    + geom_histogram(aes(x="price"), bins=50, fill="blue")
    + labs(x="Price", y="Count")
)


In [ ]:
# More bins expose finer detail but can make the histogram look noisy.
# 2) too many
(
    ggplot(data=bpp)
    + geom_histogram(aes(x="price"), bins=150, fill="blue")
    + labs(x="Price", y="Count")
)


In [ ]:
# Fewer bins combine more observations and can hide features of the distribution.
# 3) too few
(
    ggplot(data=bpp)
    + geom_histogram(aes(x="price"), bins=5, fill="blue")
    + labs(x="Price", y="Count")
)


#### Task:
 Play with the binwidth - instead of `bins=`, use `binwidth=`
 
 create 3 graphs: 1) approx ok, 2) too large binwidth, 3) too narrow binwidth
 
 discuss, what is the relation between bins and binwidth!

In [ ]:
# Specify bin width in price units rather than specifying the number of bins.
# 1) approx ok
(
    ggplot(data=bpp)
    + geom_histogram(aes(x="price"), binwidth=20, fill="blue")
    + labs(x="Price", y="Count")
)


In [ ]:
# A width of 200 creates wider, fewer bins and hides detail.
# Wider bins: fewer bars.
(
    ggplot(data=bpp)
    + geom_histogram(aes(x="price"), binwidth=200, fill="blue")
    + labs(x="Price", y="Count")
)


In [ ]:
# A width of 2 creates narrower, more numerous bins and can reveal noise.
# Narrower bins: more bars.
(
    ggplot(data=bpp)
    + geom_histogram(aes(x="price"), binwidth=2, fill="blue")
    + labs(x="Price", y="Count")
)


 Relation: they are inversely proportional

___

#### Count vs. Relative Frequency

Until now we have used count (counted the number of observations in each bin)

The other possibility is to use relative frequency instead:

You need to add `y = after_stat('density')` to the aesthetics:

In [ ]:
# Density scaling makes total bar area one; bar height itself is not a probability.
(
    ggplot(data=bpp)
    + geom_histogram(aes(x="price", y=after_stat("density")), binwidth=20, fill="blue")
    + labs(x="Price", y="Relative Frequency")
)


#### Kernel density

 Histogram or kernel density? Kernel is the smooth line instead of using bars.
 
 Now, let us name our ggplot:

In [ ]:
# Store a smoothed density curve; bandwidth controls the amount of smoothing.
my_graph = (
    ggplot(data=bpp)
    + geom_density(aes(x="price"), fill="red", alpha=0.1, bw=20)
    + labs(x="Price", y="Relative Frequency")
)


to make it visible we need to call it

In [ ]:
# Display the saved plot object without rebuilding it.
my_graph


Cool stuff about plotnine, is that we can add (later as well) new geometric object to it.

e.g. we can add a histogram:

In [ ]:
# Use density-scaled bars so the histogram and smooth curve share the same vertical
# scale.
my_graph + geom_histogram(
    aes(x="price", y=after_stat("density")), binwidth=20, fill="blue", alpha=0.4
)


#### Task
1) Do the same kernel density and histogram, but now with the price differences!

2) Add xlim(-5,5) command to ggplot! What changed?

In [ ]:
# Compare a density curve and histogram; xlim excludes out-of-range observations before
# estimation.
(
    ggplot(data=bpp)
    + geom_density(aes(x="p_diff"), fill="red", alpha=0.1, bw=0.2)
    + geom_histogram(
        aes(x="p_diff", y=after_stat("density")), binwidth=0.2, fill="blue", alpha=0.4
    )
    + xlim(-5, 5)
    + labs(x="Price Difference", y="Relative Frequency")
)


Check for high price differences

In [ ]:
# Use OR to inspect observations with very large positive or negative price differences.
bpp.loc[(bpp["p_diff"] > 500) | (bpp["p_diff"] < -500)]


Remove them

In [ ]:
# Use AND to keep differences strictly between the two cutoffs.
bpp = bpp.loc[(bpp["p_diff"] < 500) & (bpp["p_diff"] > -500)]


#### Comparing different countries via graphs

Create plot for each countries - histogram:

Note: 

  1) if you only use one type of x or y, you can put it into the `aes()` of the ggplot. Otherwise not. \
  2) use 'fill=' in `aes()`, to define different groups. 

In [ ]:
# Map country to fill to compute coloured groups; the default histogram position stacks
# them.
(
    ggplot(bpp, aes(x="p_diff", fill="COUNTRY"))
    + geom_histogram(aes(y=after_stat("density")), bins=15, alpha=0.4)
    + xlim(-4, 4)
    + labs(x="Price Difference", y="Relative Frequency")
)


Use the extra command `facet_wrap(~COUNTRY)` to create multiple plots for each country at once!

In [ ]:
# Give each country its own panel to make the distributions easier to compare.
(
    ggplot(bpp, aes(x="p_diff", fill="COUNTRY"))
    + geom_histogram(aes(y=after_stat("density")), bins=15, alpha=0.4)
    + xlim(-4, 4)
    + labs(x="Price Difference", y="Relative Frequency")
    + facet_wrap("~COUNTRY")
)


#### Task 
1) You can also use  'color=' or 'group=' instead of 'fill='.\
Compare! What is the difference?

Use color instead

In [ ]:
# Mapping country to color changes outlines rather than the bars' interior fill.
(
    ggplot(bpp, aes(x="p_diff", color="COUNTRY"))
    + geom_histogram(aes(y=after_stat("density")), bins=15, alpha=0.4)
    + xlim(-4, 4)
    + labs(x="Price Difference", y="Relative Frequency")
    + facet_wrap("~COUNTRY")
)


Use group instead

In [ ]:
# group separates calculations by country without assigning country-specific colours.
(
    ggplot(bpp, aes(x="p_diff", group="COUNTRY"))
    + geom_histogram(aes(y=after_stat("density")), bins=15, alpha=0.4)
    + xlim(-4, 4)
    + labs(x="Price Difference", y="Relative Frequency")
    + facet_wrap("~COUNTRY")
)


#### Task

1) Do the same, but use geom_density instead of geom_histogram! \
    You may play around with the xlim!\
2) Drop the `facet_wrap` command! What happens? Which graph would you use to tell your story in this case?\
What if instead of `fill` you use `color` or `group`\

 1) Density with multiple graphs

In [ ]:
# Compare smoothed country distributions in separate panels using the same bandwidth.
(
    ggplot(bpp, aes(x="p_diff", fill="COUNTRY"))
    + geom_density(alpha=0.4, bw=0.03)
    + xlim(-1, 1)
    + labs(x="Price Difference", y="Relative Frequency")
    + facet_wrap("~COUNTRY")
)


2) Density with single graphs:

In [ ]:
# Overlay density curves with coloured interiors; transparency makes overlaps visible.
# fill
(
    ggplot(bpp, aes(x="p_diff", fill="COUNTRY"))
    + geom_density(alpha=0.4, bw=0.03)
    + xlim(-1, 1)
    + labs(x="Price Difference", y="Relative Frequency")
)


In [ ]:
# Map country to line colour instead of filling the area under each curve.
# color
(
    ggplot(bpp, aes(x="p_diff", color="COUNTRY"))
    + geom_density(alpha=0.4, bw=0.03)
    + xlim(-1, 1)
    + labs(x="Price Difference", y="Relative Frequency")
)


In [ ]:
# Compute separate curves by country without colouring them differently.
# group
(
    ggplot(bpp, aes(x="p_diff", group="COUNTRY"))
    + geom_density(alpha=0.4, bw=0.03)
    + xlim(-1, 1)
    + labs(x="Price Difference", y="Relative Frequency")
)


Which graph to use: I would definitely use single density graph with color. \
    It tells the story best: there are differences between countries!

### Hypothesis testing <a class="anchor" id="ttest"></a>

[back](#TOC)

Test 1: 

H0: the average price difference between price_online - price = 0 \
HA: the avg price diff is non 0.

In [ ]:
# Load statistical test functions for the following comparisons with zero.
from scipy import stats


In [ ]:
# Test whether the mean price difference differs from zero in either direction.
stats.ttest_1samp(bpp["p_diff"], 0)


Test 2: The online prices are smaller or equal to offline prices
  
H0: price_online - price = 0 \
HA: price_online - price >  0

In [ ]:
# Use a one-sided alternative: the population mean price difference is greater than
# zero.
stats.ttest_1samp(bpp["p_diff"], 0, alternative="greater")


Test 3: The online prices are larger or equal to offline prices
    
  H0: price_online - price = 0 \
  HA: price_online - price <  0

In [ ]:
# Use the opposite one-sided alternative: the population mean is less than zero.
stats.ttest_1samp(bpp["p_diff"], 0, alternative="less")


Let us create multiple hypothesis tests: \
Check the hypothesis that online prices are the same as offline for each country!

In [ ]:
# For each country, calculate the mean, its standard error, and the non-missing sample
# size.
testing = bpp.groupby("COUNTRY").agg(
    mean_pdiff=("p_diff", "mean"),
    se_pdiff=("p_diff", "sem"),
    num_obs=("p_diff", "count"),
)
testing


Testing is easy if one understands the theory! \
t_stat: with this H0 and t-test: 

In [ ]:
# Under a zero-mean null, the t statistic is the observed mean divided by its standard
# error.
testing["t_stat"] = testing["mean_pdiff"] / testing["se_pdiff"]

testing


Calculate p-values

In [ ]:
# This computes one upper-tail probability at the absolute t statistic, not a two-sided
# p-value; a two-sided value needs a factor of two.
testing["p_val"] = stats.t.sf(abs(testing["t_stat"]), df=testing["num_obs"] - 1)

testing


Round it to 4 digits

In [ ]:
# Round the stored probabilities to four decimals; retain unrounded values when
# precision matters.
testing["p_val"] = testing["p_val"].round(4)
testing


Interpret the results for each country! \
What are the possible dangers of multiple hypothesis testing?

### Association<a class="anchor" id="association"></a>

[back](#TOC)

Relation between two variables

Association between online and retail prices: geom_point() will add dots to the graph

In [ ]:
# Each point pairs an observation's online and offline price.
(
    ggplot(bpp, aes(x="price_online", y="price"))
    + geom_point(color="red")
    + labs(x="Price online", y="Price retail")
)


You can add a line (regression line to be specific),\
  by `geom_smooth()` function. It is a great function, \
     we now focus on `method=lm` which says it is a linear relation (linear model) \
     and formula, which identifies y and x. We will discuss these more in details later.

In [ ]:
# Add a fitted straight line to describe association; the plot alone does not establish
# causation.
(
    ggplot(bpp, aes(x="price_online", y="price"))
    + geom_point(color="red")
    + geom_smooth(method="lm", formula="y ~ x", color="blue")
    + labs(x="Price online", y="Price retail")
)


#### Bin-scatter:

In many case there are too many observations for a simple graph \
  and it does not tells the story we would like to. \
One solution is to do a `bin-scatter`, which put observations into bins. \
  The simplest way to do is use "equal distances": cut x-variable's range into k equally sized bins \
    and then calculate the same observations' y-variable e.g. mean (or median). \ 
    - this is great: simple and intuitive (similar to histogram), \
      BUT it hides, how many observations are in each bin. \
        E.g. it can happen in the lowest valued bin there are many observations \
            and in the highest there is only one. \
       
  The second option is use the same number of observations in each bin. \
    This will ensure that no such problem will rise. On the other hand it is harder to compute, \
     and the width of the bins will vary along x.\

1) 'easy way': using equal distances and calculate mean for y
   use `stat_summary_bin()`

In [ ]:
# Group x into bins and plot mean y in each bin; geom=point does not draw uncertainty
# bars.
ggplot(bpp, aes(x="price_online", y="price")) + stat_summary_bin(
    fun_data="mean_se", bins=10, geom="point", color="red", size=2
)


2) 'easy way': using equal distances
   group by countries, explain facet_wrap additional inputs!

In [ ]:
# Compare binned means and fitted lines by country; free scales vary the axes between
# panels.
(
    ggplot(bpp, aes(x="price_online", y="price", color="COUNTRY"))
    + stat_summary_bin(fun_data="mean_se", bins=10, geom="point", size=2)
    + labs(x="Price online", y="Price offline", color="Country")
    + facet_wrap("~COUNTRY", scales="free", ncol=2)
    + theme(legend_position="none")
    + geom_smooth(method="lm", formula="y~x", se=False)
)


#### Bin-scatter 2 

Using percentiles instead of equally sized bins to ensure same number of observations!

As there is no built-in function for this, we need to do some work:

First, cut the y variable into 10 equally sized categories

In [ ]:
# qcut creates roughly equal-count groups, not equal-width price intervals.
bpp["price_online_10b"] = pd.qcut(bpp["price_online"], 10).values


Select these new intervals and the y-variable \
then group by the intervals and calculate some descriptive statistics! \
from these descriptive statistics we can choose which to show on the y-axis!

In [ ]:
# Summarise offline prices within each online-price group.
bpp.groupby("price_online_10b").agg(
    p_min=("price", "min"),
    p_max=("price", "max"),
    p_mean=("price", "mean"),
    p_median=("price", "median"),
    p_sd=("price", "std"),
    p_num_obs=("price", "count"),
)


Get mean of each category for x axis labels

In [ ]:
# transform repeats each group's mean online price on all rows belonging to that group.
bpp["mid_point"] = bpp.groupby("price_online_10b")["price_online"].transform("mean")


Now, calculate mean price for each midpoint

In [ ]:
# Collapse to mean offline price for each group location and return a regular DataFrame.
bs_summary = bpp.groupby("mid_point")["price"].mean().reset_index()


In [ ]:
# Inspect the summary table used to draw the binned scatterplot.
bs_summary


In [ ]:
# Plot the group summaries instead of every original observation.
(
    ggplot(bs_summary, aes(x="mid_point", y="price"))
    + geom_point()
    + geom_point(size=2, color="red")
    + labs(x="Online prices", y="Retail prices")
)


Add x and y limits to check smaller values

In [ ]:
# Restrict the displayed summary points with axis limits; out-of-range points are
# removed.
(
    ggplot(bs_summary, aes(x="mid_point", y="price"))
    + geom_point()
    + geom_point(size=2, color="red")
    + labs(x="Online prices", y="Retail prices")
    + xlim(0, 100)
    + ylim(0, 100)
)


#### Correlation and plots with factors

Often we would like to measure an association: \
covariance and correlation for mean-dependence

Covariance matrix

In [ ]:
# Covariance measures joint variation and depends on the variables' units.
bpp.filter(["price", "price_online"]).cov()


Correlation

In [ ]:
# Correlation standardises linear association to a value between -1 and 1.
bpp.filter(["price", "price_online"]).corr()


Make a correlation table, including correlation for each country

In [ ]:
# Calculate the price correlation separately within each country.
corr_table = (
    bpp.groupby("COUNTRY")["price"]
    .corr(bpp["price_online"])
    .rename("correlation")
    .reset_index()
)

corr_table


Graph to show the correlation pattern by each country: \
"reorder" will reorder the countries by their correlation

In [ ]:
# Order countries by their correlation to make the comparison easier to read.
(
    ggplot(corr_table, aes(x="correlation", y="reorder(COUNTRY, correlation)"))
    + geom_point(color="red", size=2)
    + labs(y="Countries", x="Correlation")
)


#### Task

Check the same for years and countries to check how the pattern altered!

Note: \
    1) use color for prettier output with factor \
    2) You can alter the legend labels with `color=`

In [ ]:
# Calculate separate correlations for each year-country combination.
corr_table2 = (
    bpp.groupby(["year", "COUNTRY"])["price"]
    .corr(bpp["price_online"])
    .rename("correlation")
    .reset_index()
)

corr_table2


In [ ]:
# Treat year as a category for colouring the country-level correlation points.
(
    ggplot(
        corr_table2,
        aes(x="correlation", y="reorder(COUNTRY, correlation)", color="factor(year)"),
    )
    + geom_point(size=2)
    + labs(y="Countries", x="Correlation", color="Year")
)
